# Facebook Denoiser

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from utils.clear_memory import clear_memory
from utils.batch import batch_denoise

warnings.filterwarnings('ignore')

In [ ]:
from utils.config import get_audio_files, get_output_dir

files_Pitt = get_audio_files('Pitt-origin')
out_Pitt = get_output_dir('Pitt-origin', 'Denoiser')

files_Lu = get_audio_files('Lu')
out_Lu = get_output_dir('Lu', 'Denoiser')

## Load Model

In [ ]:
from denoiser import pretrained

model_name = 'dns64'
model = pretrained.dns64()

# Device detection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

model = model.to(device)
model.eval()

target_sr = model.sample_rate  # denoiser model's built-in sample rate (typically 16000)
print(f"Model: {model_name}, Sample Rate: {target_sr}, Device: {device}")

## Denoise Function

In [ ]:
def denoise_audio(audio_path, model, device, target_sr):
    """
    Apply Facebook Denoiser for speech denoising
    """
    audio, sr = sf.read(str(audio_path))

    # Multi-channel to mono
    if len(audio.shape) == 2:
        audio = np.mean(audio, axis=1)

    # Resample to target sample rate
    if sr != target_sr:
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)

    audio = audio.astype(np.float32)

    # Normalize to [-1, 1]
    audio = np.clip(audio, -1.0, 1.0)

    # Convert to torch tensor: [batch, channels, time]
    wav = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        denoised = model(wav)

    # Back to numpy: [time]
    denoised_audio = denoised.squeeze().cpu().numpy()
    denoised_audio = np.clip(denoised_audio, -1.0, 1.0)

    return denoised_audio, target_sr

## Pitt Denoise

In [ ]:
denoise_fn = lambda p: denoise_audio(p, model, device, target_sr)

clear_memory()

batch_denoise(
    files_Pitt['Dementia'],
    out_Pitt / 'Dementia',
    denoise_fn,
    'Dementia',
)

clear_memory()

batch_denoise(
    files_Pitt['Control'],
    out_Pitt / 'Control',
    denoise_fn,
    'Control',
)

## Lu Denoise

In [ ]:
# denoise_fn = lambda p: denoise_audio(p, model, device, target_sr)

# clear_memory()

# batch_denoise(
#     files_Lu['Dementia'],
#     out_Lu / 'Dementia',
#     denoise_fn,
#     'Dementia',
# )

# clear_memory()

# batch_denoise(
#     files_Lu['Control'],
#     out_Lu / 'Control',
#     denoise_fn,
#     'Control',
# )